In [1]:
import supermag_api as sm
import pandas as pd
import numpy as np
from IPython.display import clear_output
import os
import glob
from datetime import datetime, timedelta
from tqdm import tqdm

In [2]:
def check_station_availability(user, start_dt, end_dt, target_stations):
    # 1. Calculate the 'extent' (total seconds) for the window
    extent = int((end_dt - start_dt).total_seconds())
    
    # 2. Convert start_dt to the format required by the API helper
    start_list = [start_dt.year, start_dt.month, start_dt.day, start_dt.hour, start_dt.minute]
    
    print(f"Checking availability from {start_dt} to {end_dt}...")
    
    # 3. Fetch the list of ALL stations active in this window
    status, available_stations = sm.SuperMAGGetInventory(user, start_list, extent)
    
    if not status:
        print("Error: Could not connect to SuperMAG inventory service.")
        return None

    # 4. Compare your list to the available list
    # Standardize to uppercase just in case
    available_set = set(s.upper() for s in available_stations)
    target_set = set(s.upper() for s in target_stations)
    
    ready = target_set.intersection(available_set)
    missing = target_set - available_set
    
    # 5. Report results
    print(f"\n--- Availability Report ---")
    print(f"Total Requested: {len(target_stations)}")
    print(f"Available:       {len(ready)}")
    print(f"Missing/Offline: {len(missing)}")
    
    if missing:
        print(f"Missing Stations: {sorted(list(missing))}")
    
    return sorted(list(ready))

In [ ]:
baseline='yearly'

status, df = sm.SuperMAGGetData(logon="yourid", start=[2025,2,12,0,0], extent=86400, flagstring='baseline='+baseline, station='GDH')
if not df.empty:
    for col in ['N', 'E', 'Z']:
        for system in ['nez', 'geo']:
            df[f"{col}_{system}"] = [row[system] for row in df[col]]
    df = df.drop(columns=['N', 'E', 'Z'])

    df['tval'] = pd.to_datetime(df['tval'], unit='s')
    df = df.rename(columns={'tval': 'time', 'N_nez': 'dbn_nez', 'E_nez': 'dbe_nez', 'Z_nez': 'dbz_nez', 'N_geo': 'dbn_geo', 'E_geo': 'dbe_geo', 'Z_geo': 'dbz_geo'})
    cols = ['dbn_nez','dbn_geo','dbe_nez','dbe_geo','dbz_nez','dbz_geo']
    #df[cols] = df[cols].where(df[cols] < 9999, np.nan) #

    filename = f"test_data/supermag_GDH_20250212.csv"
    df.to_csv(filename, index=False)

In [ ]:
userid = 'yourid'
stations = ['AMK', 'ATU', 'BBG', 'BJN', 'CY0', 'DMH', 'DNB', 'EUA', 'FAR', 'FHB', 'GDH', 'GHB', 'HLL', 'HOP', 'HOV', 'HRN', 'IGC', 'IQA', 'JAN',
 'KUV', 'LRV', 'LYR', 'NAL', 'NAN', 'NAQ', 'NRD', 'PGC', 'RES', 'SCO', 'SKT', 'STF', 'SUM', 'SVS',
 'T29', 'TAB', 'THL', 'UMQ', 'UPN']

baseline = 'none'

years = range(2023,2025+1)

for year in years:
    if baseline=='none':
        output_folder = f'supermag_downloads/no_baseline/{year}'
    elif baseline=='yearly':
        output_folder = f'supermag_downloads/yearly_baseline/{year}'
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    start = datetime(year, 1, 1)
    end = datetime(year, 12, 31)

    # This returns the list of stations that are safe to download
    safe_to_download = check_station_availability(userid, start, end, stations)

    for station in safe_to_download:
        for month in range(1, 12+1):
            filename = f"{output_folder}/supermag_{station}_{year}_{month}.csv"
            start = [year, month, 1, 0, 0]
            if month == 2:
                extent = 86400*28
            elif month in [4, 6, 9, 11]:
                extent = 86400*30
            elif month in [1, 3, 5, 7, 8, 10, 12]:
                extent = 86400*31
            if os.path.exists(filename):
                print(f'File for {station} in month {month} already exists')
                continue
            else:
                status, df = sm.SuperMAGGetData(logon=userid, start=start, extent=extent, flagstring='baseline='+baseline, station=station)
            if status:
                if not df.empty:
                    for col in ['N', 'E', 'Z']:
                        for system in ['nez', 'geo']:
                            df[f"{col}_{system}"] = [row[system] for row in df[col]]
                    df = df.drop(columns=['N', 'E', 'Z'])

                    df['tval'] = pd.to_datetime(df['tval'], unit='s')
                    df = df.rename(columns={'tval': 'time', 'N_nez': 'dbn_nez', 'E_nez': 'dbe_nez', 'Z_nez': 'dbz_nez', 'N_geo': 'dbn_geo', 'E_geo': 'dbe_geo', 'Z_geo': 'dbz_geo'})

                    cols = ['dbn_nez','dbn_geo','dbe_nez','dbe_geo','dbz_nez','dbz_geo']
                    if baseline=='none':
                        df[cols] = df[cols].where(df[cols] < 99999, np.nan)
                    elif baseline=='yearly':
                        df[cols] = df[cols].where(df[cols] < 9999, np.nan)

                    filename = f"{output_folder}/supermag_{station}_{year}_{month}.csv"
                    df.to_csv(filename, index=False)
                    print(f'Recieved data from station {station} in month {month}')
                else:
                    print(f'No data for station {station} in month {month}')
            else:
                print(f"Server Error (Status 0), during station {station} month {month}")
        clear_output()
        

In [3]:
pd.read_csv('supermag_downloads/no_baseline/2023/supermag_ATU_2023_1.csv')

,time,ext,iaga,dbn_nez,dbn_geo,dbe_nez,dbe_geo,dbz_nez,dbz_geo
0,2023-01-01 00:00:00,60.0,ATU,9244.308594,8101.859098,-16.357759,-4451.672507,54857.199219,54857.199219
1,2023-01-01 00:01:00,60.0,ATU,9244.767578,8104.680139,-11.319508,-4447.472939,54858.488281,54858.488281
2,2023-01-01 00:02:00,60.0,ATU,9247.435547,8108.083557,-9.105158,-4446.811007,54862.777344,54862.777344
3,2023-01-01 00:03:00,60.0,ATU,9243.015625,8103.170596,-11.262459,-4446.581944,54863.777344,54863.777344
4,2023-01-01 00:04:00,60.0,ATU,9238.652344,8099.098127,-11.772273,-4444.934786,54866.738281,54866.738281
...,...,...,...,...,...,...,...,...,...
44635,2023-01-31 23:55:00,60.0,ATU,NaN,NaN,NaN,NaN,NaN,NaN
44636,2023-01-31 23:56:00,60.0,ATU,NaN,NaN,NaN,NaN,NaN,NaN
44637,2023-01-31 23:57:00,60.0,ATU,NaN,NaN,NaN,NaN,NaN,NaN
44638,2023-01-31 23:58:00,60.0,ATU,NaN,NaN,NaN,NaN,NaN,NaN
